# Notebook 2 – PyTorch Fundamentals

## Setup

In [1]:
import numpy as np
import torch
torch.manual_seed(0)


## 1. What is PyTorch?

**Explain:** PyTorch is a Python library for building and training neural networks. It provides **tensors** (fast, GPU-capable arrays) and **automatic differentiation** (it computes gradients for us, which is what backpropagation needs).

**Analogy:** If NumPy is a calculator for arrays, PyTorch is that calculator **plus** a built-in assistant that automatically tracks every calculation so it can later tell you "how much would the result change if this input changed slightly" — exactly what's needed to train a network.

**Why/when it's needed:** Whenever we need to build a network and **train** it (not just do one-off math), because training requires gradients, and PyTorch computes those automatically.

**Implement:**

In [2]:
import torch
print("PyTorch version:", torch.__version__)

PyTorch version: 2.14.0+cpu


**Interpret:** This confirms PyTorch is installed and ready. Everything below builds on this one library.


## 2. Why PyTorch Is Used in Deep Learning

**Explain:** Three reasons: (1) **tensors run fast**, especially on GPUs, (2) **automatic differentiation** computes gradients for backpropagation without us writing calculus by hand, (3) it has a large ecosystem (TorchVision, pretrained models, etc.).

**Analogy:** Doing calculus by hand for a network with millions of parameters is like manually calculating a delivery route for a million packages — technically possible, but nobody does it by hand. PyTorch is the routing software that does it instantly.

**Why/when it's needed:** Any time the model has many parameters (even a small network has dozens) — computing gradients manually stops being practical almost immediately.

**Implement:**

In [3]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2
y.backward()          # PyTorch computes dy/dx automatically
print("dy/dx at x=3:", x.grad.item())

dy/dx at x=3: 6.0


**Interpret:** We never wrote the derivative `2x` ourselves — PyTorch computed it automatically. This automatic gradient computation is the core reason PyTorch (and similar frameworks) exist for deep learning.

## 3. Tensor

**Explain:** A **tensor** is PyTorch's core data structure — a multi-dimensional array of numbers (scalar, vector, matrix, or higher dimensions), used to hold inputs, weights, and outputs.

**Analogy:** A container that can hold numbers in different shapes — one number, a list, a table, or a stack of tables.

**Why/when it's needed:** Every single computation in a neural network (inputs, weights, activations) is stored and processed as a tensor — it's the "unit of currency" of PyTorch.

**Implement:**

In [4]:
t = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print(t)

tensor([[1., 2.],
        [3., 4.]])


**Interpret:** This 2x2 tensor could represent, e.g., 2 orders with 2 features each. Every layer in a network takes tensors in and produces tensors out.


## 4. Tensor vs NumPy Array

**Explain:** Both store multi-dimensional numeric arrays and share very similar syntax. The key difference: PyTorch tensors can **track gradients** (`requires_grad`) and can **run on a GPU** — NumPy arrays can do neither.

**Analogy:** A NumPy array is a plain notebook of numbers. A PyTorch tensor is the same notebook, but it can also (optionally) keep a running log of every calculation done on it, so it can be "replayed backwards" later.

**Why/when it's needed:** Use NumPy for general data manipulation (like in our earlier ML sprints); use PyTorch tensors specifically when the numbers need to flow through a trainable neural network.

**Implement:**

In [5]:
arr = np.array([1.0, 2.0, 3.0])
ten = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
print("NumPy array:", arr, "| has gradients?", False)
print("PyTorch tensor:", ten, "| has gradients?", ten.requires_grad)

NumPy array: [1. 2. 3.] | has gradients? False
PyTorch tensor: tensor([1., 2., 3.], requires_grad=True) | has gradients? True


**Interpret:** They look almost identical, but only the PyTorch tensor can track how it's used in later computations (`requires_grad=True`) — NumPy has no equivalent, which is exactly why we switch to PyTorch once training is involved.


## 5. Creating Tensors

**Explain:** Tensors can be created from Python lists/NumPy arrays, or generated directly (zeros, ones, random values) — useful for initializing weights before training.

**Analogy:** Setting up a blank scoreboard (zeros), a fully-filled scoreboard (ones), or handing out random starting scores (random) before a game begins.

**Why/when it's needed:** Network weights must start as *some* numbers before training — usually small random values — so tensor creation functions are used at model initialization.

**Implement:**

In [6]:
print(torch.zeros(2, 3))
print(torch.ones(2, 3))
print(torch.rand(2, 3))     # e.g. how weights are initialized before training

tensor([[0., 0., 0.],
        [0., 0., 0.]])
tensor([[1., 1., 1.],
        [1., 1., 1.]])
tensor([[0.4963, 0.7682, 0.0885],
        [0.1320, 0.3074, 0.6341]])


**Interpret:** `zeros`/`ones` give fixed starting points; `rand` gives random ones — this last one is essentially what happens (with a specific method) when a network's weights are first created, before any training happens.


## 6. Tensor Shape

**Explain:** The **shape** tells us the size along each dimension (e.g. rows x columns). It must match between layers and operations, or PyTorch will raise an error.

**Analogy:** The dimensions of a shipping box — you can't fit a box shaped 3x4 into a slot shaped 4x3 without first rotating/reshaping it.

**Why/when it's needed:** Checking shape is the #1 debugging step in deep learning — most errors are shape mismatches between layers or operations.

**Implement:**

In [7]:
batch = torch.rand(5, 3)   # 5 orders, 3 features each
print("Shape:", batch.shape)

Shape: torch.Size([5, 3])


**Interpret:** `(5, 3)` means 5 rows (samples) and 3 columns (features) — this must match what the next layer expects as its input size.


## 7. Tensor Dimensions

**Explain:** The number of dimensions (`ndim`) tells us how "deep" the nesting is: 0-D (single number), 1-D (list), 2-D (table), 3-D+ (e.g. a batch of images).

**Analogy:** A single grade (0-D), a list of grades (1-D), a gradebook table (2-D), a stack of gradebooks from different classes (3-D).

**Why/when it's needed:** Different layers expect different dimensionalities — e.g. a normal layer expects 2-D (batch x features), a convolution layer expects 4-D (batch x channels x height x width). Getting this wrong is a very common bug.

**Implement:**

In [8]:
for t in [torch.tensor(5.0), torch.tensor([1.,2.]), torch.rand(2,3), torch.rand(2,3,4)]:
    print("ndim:", t.ndim, "shape:", tuple(t.shape))

ndim: 0 shape: ()
ndim: 1 shape: (2,)
ndim: 2 shape: (2, 3)
ndim: 3 shape: (2, 3, 4)


**Interpret:** ndim grows from 0 to 3 as we go from a single number to a 3-D block — this is exactly why we must check dimensions before feeding data into a specific type of layer.

## 8. Tensor Dtype

**Explain:** The **dtype** is the type of numbers stored (e.g. `float32`, `int64`). Most neural network computations expect **floating point** tensors; mixing dtypes causes errors.

**Analogy:** Trying to add measurements in inches to measurements in centimeters without converting first — the math breaks unless the units (types) match.

**Why/when it's needed:** Real-world data often loads as integers (e.g. `Quantity`); before feeding into a network, we must convert to float — this exact issue caused the "integer data" error we hit with `PartialDependenceDisplay` earlier in this project.

**Implement:**

In [9]:
int_tensor = torch.tensor([1, 2, 3])
float_tensor = int_tensor.float()
print("int dtype:", int_tensor.dtype, "-> float dtype:", float_tensor.dtype)

int dtype: torch.int64 -> float dtype: torch.float32


**Interpret:** The same numbers, but PyTorch now treats them as decimals, ready for the weight multiplications a network performs (which require floats).


## 9. Tensor Indexing

**Explain:** **Indexing** retrieves a specific element or row using its position, just like a Python list but across multiple dimensions.

**Analogy:** Looking up a single cell in a spreadsheet by its row and column number.

**Why/when it's needed:** Used constantly to inspect specific samples, pick out a single prediction, or select a particular feature column during debugging or preprocessing.

**Implement:**

In [10]:
data = torch.tensor([[10, 20], [30, 40], [50, 60]])
print("Row 1:", data[1])
print("Element at row 2, col 0:", data[2, 0])

Row 1: tensor([30, 40])
Element at row 2, col 0: tensor(50)


**Interpret:** Indexing let us pull out exactly the piece of data we wanted — the 2nd row, or one specific value — without touching the rest of the tensor.


## 10. Tensor Slicing

**Explain:** **Slicing** retrieves a *range* of elements (not just one), using `start:stop` syntax.

**Analogy:** Photocopying pages 2 through 4 of a book, instead of just page 3.

**Why/when it's needed:** Used to split data into batches, grab a subset of features (e.g. all rows, only the first 2 columns), or take a portion of a sequence.

**Implement:**

In [11]:
print("First 2 rows:\n", data[:2])
print("All rows, column 0 only:", data[:, 0])

First 2 rows:
 tensor([[10, 20],
        [30, 40]])
All rows, column 0 only: tensor([10, 30, 50])


**Interpret:** Slicing grabbed whole ranges at once — this is exactly how we'd split a dataset into mini-batches or select specific feature columns during training.


## 11. Tensor Reshaping

**Explain:** **Reshaping** changes a tensor's shape while keeping the same total number of elements and the same underlying data — just rearranged.

**Analogy:** Rearranging 12 eggs from a 3x4 carton into a 2x6 carton — same 12 eggs, different layout.

**Why/when it's needed:** Different layers expect different shapes (e.g. flattening an image before a fully-connected layer) — reshaping bridges that mismatch.

**Implement:**

In [12]:
x = torch.arange(12)             # 12 numbers, shape (12,)
print(x.reshape(3, 4))            # same 12 numbers, laid out as 3x4

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])


**Interpret:** Same 12 values, now arranged as 3 rows x 4 columns — this reshaping is required whenever a layer downstream expects a specific shaped input.

### 11a. `reshape()`
**Explain:** Returns a tensor with the new shape; **may copy** the data if needed to do so.
**Why/when needed:** The general-purpose, safest way to change shape — use it by default.
**Implement:**

In [13]:
print(x.reshape(4, 3))

tensor([[ 0,  1,  2],
        [ 3,  4,  5],
        [ 6,  7,  8],
        [ 9, 10, 11]])


**Interpret:** Same data, now 4x3 instead of 3x4 — `reshape` handled it without us worrying about memory details.

### 11b. `view()`
**Explain:** Also changes shape, but **only works on contiguous memory** (no copying) — faster, but stricter than `reshape()`.
**Why/when needed:** Use when you know the tensor's memory layout is compatible and want a lightweight, no-copy reshape (common right after creating a tensor).
**Implement:**

In [14]:
print(x.view(2, 6))

tensor([[ 0,  1,  2,  3,  4,  5],
        [ 6,  7,  8,  9, 10, 11]])


**Interpret:** Same result as `reshape` here, but `view()` did it without copying data — it only works when possible in-place, whereas `reshape()` falls back to copying if needed.

### 11c. `flatten()`
**Explain:** Collapses a tensor down to **1 dimension**, regardless of its original shape.
**Why/when needed:** Used right before a fully-connected layer, when data coming from a multi-dimensional source (like an image) needs to become a single flat vector.
**Implement:**

In [15]:
grid = torch.rand(2, 3)
print("Before:", grid.shape, "-> After flatten:", grid.flatten().shape)

Before: torch.Size([2, 3]) -> After flatten: torch.Size([6])


**Interpret:** A 2x3 table became one long list of 6 numbers — exactly the step needed before passing image-like data into a standard linear layer.


## 12. Tensor Arithmetic

**Explain:** Tensors support element-wise math (`+`, `-`, `*`, `/`) — the operation is applied to each matching position independently.

**Analogy:** Adding two grocery lists item by item (milk+milk, eggs+eggs) rather than combining them into one long list.

**Why/when it's needed:** This is exactly how a layer adds its bias to a weighted sum, or how a loss function compares predictions to true values, element by element.

**Implement:**

In [16]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([10.0, 20.0, 30.0])
print("a + b =", a + b)
print("a * b =", a * b)

a + b = tensor([11., 22., 33.])
a * b = tensor([10., 40., 90.])


**Interpret:** Each element combined only with the element in the same position — this is the building block behind adding a bias vector, or computing per-example prediction errors.


## 13. Matrix Multiplication

**Explain:** Unlike element-wise multiplication, **matrix multiplication** (`@` or `torch.matmul`) combines rows of one matrix with columns of another — this is exactly what a neural network layer does: `output = input @ weights + bias`.

**Analogy:** Calculating a final exam score as a weighted sum of several test scores — each score is "matched and multiplied" with its own weight, then summed.

**Why/when it's needed:** Every linear/fully-connected layer in a neural network is fundamentally a matrix multiplication — this is the single most common operation in deep learning.

**Implement:**

In [17]:
inputs = torch.tensor([[1.0, 2.0]])           # 1 sample, 2 features
weights = torch.tensor([[0.5], [0.5]])         # 2 features -> 1 output
print("Layer output (inputs @ weights):", inputs @ weights)

Layer output (inputs @ weights): tensor([[1.5000]])


**Interpret:** `1*0.5 + 2*0.5 = 1.5` — this single matrix multiplication is exactly what happens inside `nn.Linear`, just done manually here to see the mechanics.


## 14. Broadcasting

**Explain:** **Broadcasting** lets PyTorch perform operations between tensors of *different but compatible* shapes, by automatically expanding the smaller one — without actually copying data.

**Analogy:** Adding the same flat tax (one number) to every item's price in a whole shopping cart, without manually repeating that tax value for every single item.

**Why/when it's needed:** This is exactly how a **bias** (one number per neuron) gets added to an entire batch of outputs (many rows), or how one scalar (like a learning rate) is applied across an entire tensor.

**Implement:**

In [18]:
batch_outputs = torch.tensor([[1.0, 2.0], [3.0, 4.0]])   # 2 samples, 2 outputs
bias = torch.tensor([10.0, 20.0])                          # 1 bias per output
print(batch_outputs + bias)

tensor([[11., 22.],
        [13., 24.]])


**Interpret:** The single bias row was automatically applied to *every* sample in the batch — we didn't need to manually repeat it. This is exactly how bias addition works inside every layer of a network.


## 15. CPU vs GPU Concept

**Explain:** A **CPU** does a few calculations very flexibly; a **GPU** does thousands of simple calculations **in parallel** — ideal for the repeated matrix multiplications in deep learning, making training much faster.

**Analogy:** A CPU is one highly skilled chef cooking dishes one at a time. A GPU is a thousand cooks each making one identical simple dish simultaneously — much faster for large, repetitive batches of work.

**Why/when it's needed:** For small tasks (like our tiny examples), CPU is fine. For large networks/datasets (e.g. training on millions of images), a GPU can be 10-100x faster.

**Implement:**

In [19]:
print("GPU available:", torch.cuda.is_available())
print("Device we'll use:", "cuda" if torch.cuda.is_available() else "cpu")

GPU available: False
Device we'll use: cpu


**Interpret:** This machine has no GPU available, so PyTorch will run everything on CPU — fine for our small examples, but this exact check is what real training scripts use to automatically use a GPU when one exists.


## 16. Device Management

**Explain:** Tensors and models live on a specific **device** (`'cpu'` or `'cuda'`). Operations between tensors require them to be **on the same device** — mixing them raises an error.

**Analogy:** Two people can only hand objects to each other directly if they're in the same room — if one is in a different room (device), the item must first be moved there.

**Why/when it's needed:** Before training on a GPU, both the model **and** the data must be explicitly moved to that device — a very common real-world bug is forgetting to move one of the two.

**Implement:**

In [20]:
device = "cuda" if torch.cuda.is_available() else "cpu"
x = torch.tensor([1.0, 2.0]).to(device)
print("Tensor is now on device:", x.device)

Tensor is now on device: cpu


**Interpret:** The tensor now explicitly lives on `cpu` here (it would say `cuda:0` if a GPU were available). Every tensor **and** the model must be moved with the same `.to(device)` call, or PyTorch will throw a device-mismatch error.


## 17. Converting NumPy ↔ PyTorch

**Explain:** `torch.from_numpy()` converts a NumPy array into a tensor; `.numpy()` converts a tensor back. This is needed because data loading (pandas/NumPy) and modeling (PyTorch) are separate ecosystems that must be bridged.

**Analogy:** Translating a document between two languages so two different people (NumPy-based tools and PyTorch-based tools) can both understand the same information.

**Why/when it's needed:** After loading and cleaning data with pandas/NumPy (as in our previous sprints), we must convert it to a tensor before feeding it into a PyTorch model — and convert predictions back to NumPy for use with scikit-learn metrics.

**Implement:**

In [21]:
numpy_array = np.array([1.0, 2.0, 3.0])
tensor_from_numpy = torch.from_numpy(numpy_array)
back_to_numpy = tensor_from_numpy.numpy()
print(type(numpy_array), "->", type(tensor_from_numpy), "->", type(back_to_numpy))

<class 'numpy.ndarray'> -> <class 'torch.Tensor'> -> <class 'numpy.ndarray'>


**Interpret:** The same data moved from NumPy to PyTorch and back again — this round trip is exactly what we did in Notebook 1 (`X_train` came from pandas/NumPy, then became a tensor for the network) and what we do again to compute metrics like `roc_auc_score` (which expects NumPy arrays).

## 18. `torch.Tensor`

**Explain:** `torch.Tensor` is the underlying **class** for all tensors — `torch.tensor(...)` (lowercase) is the recommended **function** to create one. Both produce the same type of object, but the lowercase function is preferred because it infers dtype more safely.

**Why/when it's needed:** Knowing this avoids confusion when reading PyTorch documentation or error messages, which refer to the `torch.Tensor` class.

**Implement:**

In [22]:
a = torch.tensor([1, 2, 3])     # recommended
print(type(a), "is instance of torch.Tensor:", isinstance(a, torch.Tensor))

<class 'torch.Tensor'> is instance of torch.Tensor: True


**Interpret:** Every tensor we create — no matter how — is an instance of the `torch.Tensor` class. We use the lowercase `torch.tensor()` function in practice because it's safer about inferring the right dtype.


## 19. `requires_grad`

**Explain:** Setting `requires_grad=True` tells PyTorch to **track every operation** done on that tensor, so it can later compute gradients (derivatives) automatically via backpropagation.

**Analogy:** Turning on a car's dashcam before a trip — it silently records everything so that, if needed later, you can replay exactly what happened and compute what led to the outcome.

**Why/when it's needed:** Only tensors we intend to **train** (like model weights) need this — data inputs (like `X`) usually don't need it, since we don't want to change the data itself, only the weights.

**Implement:**

In [23]:
w = torch.tensor(2.0, requires_grad=True)
loss = (w - 5) ** 2
loss.backward()
print("Gradient of loss w.r.t. w:", w.grad.item())

Gradient of loss w.r.t. w: -6.0


**Interpret:** PyTorch automatically computed how much `loss` would change if `w` changed slightly (the gradient is 2*(2-5) = -6). This exact mechanism — enabled only by `requires_grad=True` — is what powers backpropagation and weight updates during training.


## 20. `torch.no_grad()`

**Explain:** A context manager that **temporarily disables** gradient tracking. Used whenever we only want to use a model (e.g. for prediction/evaluation), not train it.

**Analogy:** Turning the dashcam off during a casual drive where you don't need a recording — it saves memory and effort since you're not going to analyze this trip later.

**Why/when it's needed:** Tracking gradients uses extra memory and computation. During prediction/evaluation (no `.backward()` needed), wrapping code in `torch.no_grad()` makes it faster and uses less memory.

**Implement:**

In [24]:
w = torch.tensor(2.0, requires_grad=True)
with torch.no_grad():
    prediction = w * 3
print("requires_grad tracked here?", prediction.requires_grad)

requires_grad tracked here? False


**Interpret:** Even though `w` normally tracks gradients, inside `torch.no_grad()` the result does not — exactly what we want when just using a trained model to make predictions, as we did with `model(X_test)` in Notebook 1.